# CIFAR-10 NC Baseline: ResNet-20

Tests whether the feature norm threshold (~1.06) found on MNIST
generalises to CIFAR-10 with ResNet-20.

**Setup (from Jacot et al. 2025 ICLR CIFAR-10 experiments):**
- ResNet-20, SGD momentum=0.9, lr=0.1, wd=1e-3
- No data augmentation (NC requires clean within-class features)
- Phase 1: CE loss, 200 epochs, MultiStepLR decay at 100/150
- Phase 2: MSE loss, 600 epoch budget, MultiStepLR at 300/450
- 3 seeds

**Outputs:** `cifar10_summary.csv`, per-seed CSVs, `fig_cifar10_nc.png`

**Est. runtime: ~90 min on T4** (3 seeds × ~30 min)

**Settings → T4 GPU → Save & Run All**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [8]:
pwd

'/content'

In [1]:
import torch, torchvision, time
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = '/kaggle/working/'
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
print(f'GPU: {torch.cuda.get_device_name(0)}  |  PyTorch: {torch.__version__}')

# ── What we're testing ────────────────────────────────────────────────────
# MNIST baseline feature norm at T_NC: 1.063 (seed 0, depth 5, ReLU)
# Question: does CIFAR-10 with ResNet-20 collapse to a similar fn value?
# Setup mirrors Jacot et al. 2025 (ICLR) CIFAR-10 experiments:
#   ResNet-20, SGD, wd=1e-3, cosine LR, MSE loss in Phase 2
# Two-phase protocol: CE 200ep -> MSE until NC1<0.01 or 600ep budget
MNIST_FN_BASELINE = 1.063
print(f'MNIST baseline fn at T_NC: {MNIST_FN_BASELINE}')


GPU: NVIDIA A100-SXM4-80GB  |  PyTorch: 2.10.0+cu128
MNIST baseline fn at T_NC: 1.063


In [2]:
# CIFAR-10: 50k train / 10k test, 32x32 RGB, 10 classes
# NO data augmentation — NC literature uses clean images for NC metrics
# (augmentation would make within-class variability artificially high)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))
])
trainset = torchvision.datasets.CIFAR10('/kaggle/working/data',
    train=True,  download=True, transform=transform)
testset  = torchvision.datasets.CIFAR10('/kaggle/working/data',
    train=False, download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=128, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=256, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'CIFAR-10: {len(trainset):,} train / {len(testset):,} test')
print(f'Train batches: {len(train_loader)} x 128')


100%|██████████| 170M/170M [00:13<00:00, 13.1MB/s]


CIFAR-10: 50,000 train / 10,000 test
Train batches: 391 x 128


In [3]:
# ResNet-20 for CIFAR-10 (He et al. 2016, exact CIFAR variant)
# depth = 6n+2 where n=3 -> 20 layers
# Feature dim = 64 (avgpool output)

class BasicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride,
                               padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
        self.skip  = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.skip = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c))
    def forward(self, x):
        return F.relu(
            self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))
            + self.skip(x))

class ResNet20(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1  = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(16)
        self.layer1 = self._make(16, 16, 3, 1)
        self.layer2 = self._make(16, 32, 3, 2)
        self.layer3 = self._make(32, 64, 3, 2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.fc     = nn.Linear(64, num_classes)
        self._feats = None
        self.pool.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.flatten(1).detach()))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make(self, in_c, out_c, n, stride):
        layers = [BasicBlock(in_c, out_c, stride)]
        for _ in range(n-1):
            layers.append(BasicBlock(out_c, out_c, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer3(self.layer2(self.layer1(x)))
        return self.fc(self.pool(x).flatten(1))

    def get_features(self, x):
        self(x); return self._feats

    def get_classifier_weights(self):
        return self.fc.weight.detach()

# Sanity check
m = ResNet20().to(DEVICE)
x = torch.randn(4, 3, 32, 32).to(DEVICE)
f = m.get_features(x)
p = sum(q.numel() for q in m.parameters())/1e6
print(f'ResNet-20: feats={tuple(f.shape)}  params={p:.3f}M')
del m, x, f


ResNet-20: feats=(4, 64)  params=0.272M


In [4]:
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE)).cpu())
        ll.append(y)
    H = torch.cat(fl).float(); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T@(H[Y==c]-mu_c[c]) for c in range(K))/len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw)/torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool)
    nc2  = (cos[mask]-(-1./(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().cpu(), dim=1)
    nc3  = (1-(Mn*Wn).sum(1).mean()).item()
    return {'nc1':nc1,'nc2':nc2,'nc3':nc3,
            'feat_norm':H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(DEVICE),y.to(DEVICE)
            correct+=(model(x).argmax(1)==y).sum().item(); total+=len(y)
    return correct/total

print('NC metrics ready.')


NC metrics ready.


In [5]:
def run_cifar(model, name, lr=0.1, wd=1e-3,
              phase1=200, phase2=600, nc_every=10):
    """
    Two-phase training for CIFAR-10 NC.
    Phase 1: SGD + CE loss -> reach 99% train acc
    Phase 2: SGD + MSE loss -> drive NC1 collapse

    Key differences from MNIST:
    - SGD (not Adam): standard for ResNet/CIFAR
    - lr=0.1 with MultiStep decay (standard CIFAR schedule)
    - wd=1e-3: stronger than MNIST, needed for ResNet NC
    - No augmentation: NC requires clean within-class features
    """
    model = model.to(DEVICE)
    K = 10; rows = []; terminal = False; t0 = time.time()

    for phase, loss_fn, n_ep, milestones in [
        (1, 'ce',  phase1, [100, 150]),
        (2, 'mse', phase2, [300, 450]),
    ]:
        opt = torch.optim.SGD(model.parameters(), lr=lr,
                              momentum=0.9, weight_decay=wd,
                              nesterov=True)
        sched = torch.optim.lr_scheduler.MultiStepLR(
                    opt, milestones=milestones, gamma=0.1)
        off = phase1 if phase == 2 else 0

        for ep_l in range(1, n_ep+1):
            ep = off + ep_l
            model.train()
            for x, y in train_loader:
                x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                if loss_fn == 'mse':
                    loss = F.mse_loss(logits, F.one_hot(y,K).float())
                else:
                    loss = F.cross_entropy(logits, y)
                loss.backward(); opt.step()
            sched.step()

            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr = evaluate(model, train_loader)
                te = evaluate(model, test_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True
                    print(f'  [{name}] Terminal phase at epoch {ep}')
                nc = compute_nc(model, train_loader) if terminal else \
                     {'nc1':None,'nc2':None,'nc3':None,'feat_norm':None}
                rows.append({'epoch':ep,'phase':phase,'train':tr,'test':te,**nc})
                nc1s = f"{nc['nc1']:.5f}" if nc['nc1'] is not None else 'N/A'
                fns  = f"{nc['feat_norm']:.3f}" if nc['feat_norm'] else 'N/A'
                print(f'  ep={ep:>4} tr={tr:.4f} te={te:.4f} '
                      f'nc1={nc1s} fn={fns} t={(time.time()-t0)/60:.1f}m')
                if nc['nc1'] is not None and nc['nc1'] < 0.01:
                    fn_val = nc['feat_norm']
                    print(f'  *** T_NC={ep}  fn={fn_val:.4f}  '
                          f'MNIST baseline={MNIST_FN_BASELINE}')
                    return pd.DataFrame(rows), ep, fn_val
    return pd.DataFrame(rows), None, None

print('run_cifar() ready.')
print('SGD + MultiStepLR, no augmentation, phase1=CE phase2=MSE')


run_cifar() ready.
SGD + MultiStepLR, no augmentation, phase1=CE phase2=MSE


In [6]:
# Run 3 seeds for robustness
# Expected: terminal phase ~ep 100-150, T_NC somewhere in phase2
# Expected runtime: ~30 min per seed on T4 (mini-batch SGD, 50k samples)
# Total: ~90 min for 3 seeds

cifar_results = []
for seed in range(3):
    print(f'\n=== CIFAR-10 ResNet-20 seed={seed} ===')
    torch.manual_seed(seed)
    model = ResNet20()
    df, t_nc, fn = run_cifar(model, f'ResNet20-s{seed}',
                             lr=0.1, wd=1e-3,
                             phase1=200, phase2=600)
    df.to_csv(f'{SAVE_DIR}cifar10_s{seed}.csv', index=False)
    cifar_results.append({'seed':seed,'T_NC':t_nc,'fn':fn,
                          'test_acc':df.test.iloc[-1]})
    status = f'T_NC={t_nc}  fn={fn:.4f}' if t_nc else \
             f'DNF  final_nc1={df.dropna(subset=["nc1"]).nc1.iloc[-1]:.5f}'
    print(f'  => {status}')

df_cifar = pd.DataFrame(cifar_results)
df_cifar.to_csv(SAVE_DIR + 'cifar10_summary.csv', index=False)
print('\n=== CIFAR-10 SUMMARY ===')
print(df_cifar.to_string())

confirmed = df_cifar.dropna(subset=['fn'])
if len(confirmed):
    fns = confirmed.fn.values
    print(f'\nfn at T_NC: {[round(f,4) for f in fns]}')
    print(f'Mean: {np.mean(fns):.4f}  Std: {np.std(fns):.4f}')
    print(f'MNIST baseline: {MNIST_FN_BASELINE}')
    diff = abs(np.mean(fns) - MNIST_FN_BASELINE)
    print(f'Difference from MNIST: {diff:.4f}')
    if diff < 0.2:
        print('fn is CONSISTENT with MNIST baseline!')
    else:
        print(f'fn differs from MNIST by {diff:.4f} — threshold is dataset/arch dependent')



=== CIFAR-10 ResNet-20 seed=0 ===
  ep=  10 tr=0.7266 te=0.6966 nc1=N/A fn=N/A t=1.2m
  ep=  20 tr=0.7441 te=0.7039 nc1=N/A fn=N/A t=2.3m
  ep=  30 tr=0.8233 te=0.7808 nc1=N/A fn=N/A t=3.5m
  ep=  40 tr=0.8255 te=0.7818 nc1=N/A fn=N/A t=4.6m
  ep=  50 tr=0.7908 te=0.7512 nc1=N/A fn=N/A t=5.8m
  ep=  60 tr=0.8095 te=0.7654 nc1=N/A fn=N/A t=6.9m
  ep=  70 tr=0.7943 te=0.7525 nc1=N/A fn=N/A t=8.1m
  ep=  80 tr=0.7869 te=0.7440 nc1=N/A fn=N/A t=9.3m
  ep=  90 tr=0.8243 te=0.7780 nc1=N/A fn=N/A t=10.4m
  ep= 100 tr=0.7385 te=0.7001 nc1=N/A fn=N/A t=11.6m
  [ResNet20-s0] Terminal phase at epoch 110
  ep= 110 tr=0.9985 te=0.8787 nc1=0.22937 fn=6.046 t=12.8m
  ep= 120 tr=0.9660 te=0.8455 nc1=0.27324 fn=5.999 t=14.1m
  ep= 130 tr=0.9830 te=0.8639 nc1=0.24943 fn=5.985 t=15.3m
  ep= 140 tr=0.9712 te=0.8471 nc1=0.24661 fn=5.961 t=16.6m
  ep= 150 tr=0.9847 te=0.8525 nc1=0.23086 fn=5.896 t=17.9m
  ep= 160 tr=1.0000 te=0.8851 nc1=0.13961 fn=5.962 t=19.1m
  ep= 170 tr=1.0000 te=0.8848 nc1=0.12629 fn=

In [7]:
if len(df_cifar.dropna(subset=['fn'])) == 0:
    print('No T_NC reached — check per-seed CSVs for NC1 trends')
    # Show NC1 trajectory from seed 0
    try:
        df0 = pd.read_csv(f'{SAVE_DIR}cifar10_s0.csv')
        nc0 = df0.dropna(subset=['nc1'])
        if len(nc0):
            plt.figure(figsize=(8,4))
            plt.semilogy(nc0.epoch, nc0.nc1)
            plt.axhline(0.01, ls='--', color='black')
            plt.xlabel('Epoch'); plt.ylabel('NC1 (log)')
            plt.title('CIFAR-10 NC1 trajectory (seed 0)')
            plt.savefig(SAVE_DIR+'fig_cifar10_nc1.png', dpi=120)
            plt.show()
            print(f'Final NC1: {nc0.nc1.iloc[-1]:.5f}')
            print(f'Final fn:  {nc0.feat_norm.iloc[-1]:.4f}')
            slope = (nc0.nc1.iloc[-1]-nc0.nc1.iloc[-6])/(nc0.epoch.iloc[-1]-nc0.epoch.iloc[-6])
            print(f'NC1 slope (last 5 pts): {slope:.7f}/ep')
            if slope < 0:
                est = (nc0.nc1.iloc[-1]-0.01)/(-slope)
                print(f'Est. additional epochs to NC1=0.01: {est:.0f}')
    except: pass
else:
    plt.rcParams.update({'font.family':'serif','font.size':11,
        'axes.spines.top':False,'axes.spines.right':False})

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    colors = ['#2196F3','#4CAF50','#F44336']
    for seed in range(3):
        try:
            df_s = pd.read_csv(f'{SAVE_DIR}cifar10_s{seed}.csv')
            nc_s = df_s.dropna(subset=['nc1'])
            if len(nc_s):
                axes[0].plot(df_s.epoch, df_s.train,
                             color=colors[seed], lw=2, label=f's{seed}')
                axes[0].plot(df_s.epoch, df_s.test,
                             color=colors[seed], lw=2, ls='--')
                axes[1].semilogy(nc_s.epoch, nc_s.nc1,
                                 color=colors[seed], lw=2, label=f's{seed}')
                axes[2].plot(nc_s.epoch, nc_s.feat_norm,
                             color=colors[seed], lw=2, label=f's{seed}')
        except: pass

    axes[0].axhline(0.99, color='gray', ls=':', lw=1)
    axes[0].set(xlabel='Epoch', ylabel='Accuracy', title='(a) Accuracy')
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.25)

    axes[1].axhline(0.01, color='black', ls=':', lw=1.2, label='NC1=0.01')
    axes[1].set(xlabel='Epoch', ylabel='NC1 (log)', title='(b) NC1 collapse')
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.25)

    # Mark fn at T_NC
    for _, row in df_cifar.dropna(subset=['fn']).iterrows():
        s = int(row.seed)
        axes[2].axhline(row.fn, color=colors[s], ls='--', lw=1,
                        label=f's{s} fn={row.fn:.3f}')
    axes[2].axhline(MNIST_FN_BASELINE, color='black', ls=':',
                    lw=1.5, label=f'MNIST={MNIST_FN_BASELINE}')
    axes[2].set(xlabel='Epoch', ylabel='Feature norm',
                title='(c) Feature norm vs MNIST baseline')
    axes[2].legend(fontsize=8); axes[2].grid(alpha=0.25)

    fig.suptitle('CIFAR-10 ResNet-20: NC Dynamics | '
                 'SGD wd=1e-3 | Two-phase CE->MSE',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(SAVE_DIR + 'fig_cifar10_nc.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: fig_cifar10_nc.png')
    print('Saved: cifar10_summary.csv')


Saved: fig_cifar10_nc.png
Saved: cifar10_summary.csv
